# Single-neuron multimodal source-to-field/readout workflow

This tutorial demonstrates the jaxfne source-to-field/readout pipeline on a minimal single Izhikevich neuron.

**What you will learn:**
- Configure a single-neuron network
- Run a JAX-native simulation
- Apply the eight standard proxy readout operators
- Inspect the output bundle (manifest, probe reports, metrics)
- Understand claim-status metadata (computational scaffold, not empirically validated)

**Estimated time:** 5–10 minutes

In [ ]:
!pip install jaxfne

In [ ]:
import jaxfne
print(f"jaxfne version: {jaxfne.__version__}")

## Imports

Import JAX and jaxfne components.

In [ ]:
import jax
import jax.numpy as jnp
from jaxfne.fields import (
    spk_probe,
    vm_probe,
    source_probe,
    lfp_proxy_probe,
    csd_proxy_probe,
    eeg_proxy_probe,
    meg_proxy_probe,
    emm_proxy_probe,
)

# Verify JAX is available
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## Configuration

Set up a minimal single-neuron network with an Izhikevich emitter and laminar field domain.

In [ ]:
# Create minimal single-neuron configuration
# Single excitatory neuron, 100ms simulation, 0.1ms dt
cfg = (
    jaxfne.configuration()
    .network(
        name="single_neuron",
        kind="isolated_neuron",
        n=1,
        cell_types={"E": 1.0},
    )
    .emitter(family="izhikevich", preset="cortical_eig")
    .field(
        domain="laminar_column",
        conductivity="proxy",
        boundary="declared_proxy",
        gauge="mean_zero",
    )
    .probe(
        name="multimodal_single_neuron",
        modes=[
            "spikes",
            "V_m",
            "source",
            "phi_e",
            "J_e",
            "CSD",
            "LFP",
        ],
    )
)

# Construct the JAX-native model
model = jaxfne.construct(cfg)
print("Model constructed successfully.")

## Run Simulation

Simulate 100 milliseconds of neural activity with 0.1ms timesteps (1000 total timesteps).
The seed ensures reproducibility.

In [ ]:
# Define simulation parameters
sim = jaxfne.simulation(duration_ms=100.0, dt_ms=0.1, seed=42)

# Run the simulation
signals = model.simulate(sim)

print(f"Simulation complete.")
print(f"Spike matrix shape: {signals.spikes.shape}")
print(f"Voltage shape: {signals.V_m.shape}")
if hasattr(signals, 'field') and signals.field is not None:
    print(f"LFP proxy shape: {signals.field.lfp_proxy.shape}")

## Apply Readout Operators

Apply all eight standard proxy readout operators to the simulation signals.
These operators extract different modalities and summary statistics.

In [ ]:
# Apply all eight probe operators
# SPK and Vm are always available
spk_readout = spk_probe(signals.spikes)
vm_readout = vm_probe(signals.V_m)

# Source and field-derived operators (LFP, CSD) depend on field existence
source_readout = (
    source_probe(signals.sources[0]) if signals.sources is not None else None
)
lfp_readout = (
    lfp_proxy_probe(signals.field.lfp_proxy)
    if signals.field is not None
    else None
)
csd_readout = (
    csd_proxy_probe(signals.field.csd_proxy)
    if signals.field is not None
    else None
)

# EEG, MEG, EMM operators use field signal (or fallback to Vm)
field_signal = (
    signals.field.lfp_proxy if signals.field is not None else signals.V_m
)
eeg_readout = eeg_proxy_probe(field_signal)
meg_readout = meg_proxy_probe(field_signal)
emm_readout = emm_proxy_probe(field_signal)

# Construct probe_report from all operators
probe_report = {
    "spk": spk_readout.report,
    "vm": vm_readout.report,
}
if source_readout is not None:
    probe_report["source"] = source_readout.report
if lfp_readout is not None:
    probe_report["lfp_proxy"] = lfp_readout.report
if csd_readout is not None:
    probe_report["csd_proxy"] = csd_readout.report

probe_report.update({
    "eeg_proxy": eeg_readout.report,
    "meg_proxy": meg_readout.report,
    "emm_proxy": emm_readout.report,
})

print(f"All eight probe operators applied:")
for op_name in probe_report.keys():
    print(f"  ✓ {op_name}")

## Inspect Output Bundle

Examine the reproducible output bundle: manifest (metadata), probe reports (operator status), and metrics (signal summary).

In [ ]:
# Get the manifest (full pipeline metadata)
manifest = model.manifest(signals=signals)

# Compute basic signal metrics
spk_rate = float(jnp.mean(jnp.sum(signals.spikes, axis=1)))
vm_mean = float(jnp.mean(signals.V_m))
vm_std = float(jnp.std(signals.V_m))

metrics = {
    "spike_rate_per_timestep": spk_rate,
    "Vm_mean_mV": vm_mean,
    "Vm_std_mV": vm_std,
}

# Display manifest keys
print("Manifest keys (pipeline metadata):")
manifest_keys = list(manifest.keys())[:5]
for key in manifest_keys:
    print(f"  - {key}")
if len(manifest.keys()) > 5:
    print(f"  ... and {len(manifest.keys()) - 5} more")

# Display signal metrics
print(f"\nSignal metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.6f}")

## Claim-Status Metadata

Verify the claim-status metadata from the manifest. These fields indicate what validation is (or is not) present.
All readouts remain computational proxies: no biological mechanism claims, no empirical validation.

In [ ]:
# Extract and display claim-status metadata
claim_status = {
    "claim_level": manifest.get("claim_level"),
    "field_claim_level": manifest.get("field_claim_level"),
    "field_solver_status": manifest.get("field_solver_status"),
    "source_calibration_status": manifest.get("source_calibration_status"),
    "physical_amplitude_claim_allowed": manifest.get("physical_amplitude_claim_allowed"),
}

print(f"Claim-status metadata (frozen, immutable):")
for key, value in claim_status.items():
    print(f"  {key}: {value}")

print(f"\n✓ Physical amplitude claims: allowed = {claim_status['physical_amplitude_claim_allowed']}")
print(f"✓ All eight operators executed successfully.")
print(f"✓ Output bundle is JSON-serializable and reproducible.")
print(f"\nTruth status: Computational scaffold, not empirically validated.")

## Summary

You have completed the single-neuron multimodal source-to-field/readout workflow:

1. **Configuration:** Defined a minimal one-neuron network with Izhikevich emitter and laminar field
2. **Simulation:** Ran 100ms of JAX-native neural dynamics (1000 timesteps)
3. **Readouts:** Applied eight standard proxy operators (SPK, Vm, source, LFP-proxy, CSD-proxy, EEG-proxy, MEG-proxy, EMM-proxy)
4. **Inspection:** Examined manifest, metrics, and claim-status metadata

### Next steps

- **[Tutorial 2: Two-neuron E/I](02_two_neuron_ei.md)** — Add coupling and recurrent dynamics
- **[Guides: Probe operators](../guides/probe_operators.md)** — Details on each readout operator
- **[API Reference](../api/index.md)** — Full function and class documentation
- **[Tensor-field workflows](../guides/tensor_field_workflows.md)** — Deep dive on field mathematics

### Questions?

File an issue or discuss on GitHub: https://github.com/HNXJ/jaxfne